<!-- GENERATED from diff-hist/docs/public/regional_details_dashboard.md by tools/gen_public_docs.py — edit the source, not here. -->

# Regional Details Dashboard

Detailed performance distributions for the cross-product of selected client ISPs
and M-Lab servers near an anchor metro. Plot lines that lie together mean uniform
service; lines that spread apart reveal servers that perform better or worse for
some users.

The **Anchor Metro** defines the region: it preselects the M-Lab servers within
the chosen **radius** and ranks the client ISPs by traffic in that metro. A
combined PDF + CDF chart is then shown for every selected server crossed with
every selected client ISP, with test counts in the legend. See the paper or
slides for how differences in these plots reveal asymmetric mid-path routing
that can adversely affect users.

For information about Differential Histograms and how they expose anomalies in Internet mid-paths
see the **[project overview](https://annealing.mattmathis.net/differential-histograms/)**.

See **[complete](https://annealing.mattmathis.net/differential-histograms/regional_details_dashboard)** Regional Details Dashboard documentation.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "datasource",
    "type": "datasource",
    "label": "",
    "description": "Backend credentials",
    "hide": 2,
    "multi": false,
    "options": [],
    "current": { "value": "P116E76923C5457A4" },
    "query_sql": "grafana-bigquery-datasource"
  },
  {
    "name": "dataset",
    "type": "constant",
    "label": "",
    "description": "Path to BQ resources not inherited from credentials",
    "hide": 2,
    "multi": false,
    "options": [],
    "current": {},
    "query_sql": "mlab-collaboration.mm_preproduction"
  },
  {
    "name": "method",
    "type": "custom",
    "label": "",
    "description": "Select processing stack version",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "cached", "value": "cached" },
      { "text": "live", "value": "live" },
      { "text": "live-DS16", "value": "live-DS16" }
    ],
    "current": { "value": "cached" },
    "query_sql": "cached, live, experimental"
  },
  {
    "name": "anchor",
    "type": "query",
    "label": "Anchor metro",
    "description": "Used to select M-Lab servers and Client ISPs to be evaluated in the  region. ",
    "hide": 0,
    "multi": false,
    "options": [],
    "current": { "value": "gru" },
    "query_sql": "# From: 2024-12-20 Prototype cached site picker  \n\nSELECT\n  FORMAT('%t %t %t, %t', metro, ContinentCode, City, CountryCode) AS text,\n  metro AS value,\nFROM (\n  SELECT\n    REGEXP_EXTRACT(site, '^([a-z]{3})') AS metro,\n    ANY_VALUE(ContinentCode) AS ContinentCode,\n    ANY_VALUE(City) AS City,\n    ANY_VALUE(CountryCode) AS CountryCode,\n  FROM `${dataset}.cached_metadata`\n  GROUP BY metro\n  ORDER BY  ContinentCode, CountryCode, metro\n)\n"
  },
  {
    "name": "radius",
    "type": "custom",
    "label": "Radius (kM)",
    "description": "Radius from the anchor metro for selecting servers",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "100", "value": "100" },
      { "text": "500", "value": "500" }
    ],
    "current": { "value": "100" },
    "query_sql": "1, 100, 500"
  },
  {
    "name": "region",
    "type": "query",
    "label": "Servers",
    "description": "Select M-Lab sites within the specified radius of the anchor metro.",
    "hide": 0,
    "multi": true,
    "options": [],
    "current": {
      "value": [
        "gru02",
        "gru03",
        "gru06"
      ]
    },
    "query_sql": " SELECT\n  # FORMAT ('TEXT=\"%t (%t) %t\" VALUE=%t', site, ANY_VALUE(ROUND(distance)), ANY_VALUE(ASName), site) AS LongText,\n  site AS value,\n  FORMAT ('%t (%t %t) %t', site, ANY_VALUE(ROUND(distance)), ANY_VALUE(CountryCode), ANY_VALUE(ASName)) AS text\n  FROM (\n    SELECT\n    ST_DISTANCE(\n      ( SELECT ANY_VALUE(ST_GEOGPOINT(Longitude, Latitude))\n        FROM `${dataset}.cached_metadata`\n        WHERE  REGEXP_CONTAINS ( site, \"^(${anchor:regex})\" )),\n      ST_GEOGPOINT(Longitude, Latitude)\n    ) / 1000.0 AS distance,\n    *\n    -- Uses ${datasource}  Implicitly\n    FROM `mlab-collaboration.mm_preproduction.cached_metadata`\n    ORDER BY distance, site\n)\nWHERE distance < ${radius}\nGROUP BY site\n",
    "default_select": "all"
  },
  {
    "name": "ISPcount",
    "type": "custom",
    "label": "Client Rows",
    "description": "Number of Client ISPs to preprocess.   It is best to include extras, beyond the Client ISP selectors.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "0", "value": "0" },
      { "text": "1", "value": "1" },
      { "text": "2", "value": "2" },
      { "text": "3", "value": "3" },
      { "text": "4", "value": "4" },
      { "text": "5", "value": "5" },
      { "text": "10", "value": "10" },
      { "text": "20", "value": "20" },
      { "text": "50", "value": "50" }
    ],
    "current": { "value": "10" },
    "query_sql": "0,1,2,3,4,5,10,20,50"
  },
  {
    "name": "extra_rows",
    "type": "textbox",
    "label": "Extra rows",
    "description": "Added to Client Rows when fetching from BQ; shown only when the selected servers span multiple metros.",
    "hide": 0,
    "multi": false,
    "options": [],
    "current": { "value": "0" },
    "query_sql": null
  },
  {
    "name": "ClientISP",
    "type": "query",
    "label": "Client ISPs",
    "description": "Select Client ISPs of interest.   ",
    "hide": 0,
    "multi": true,
    "options": [],
    "current": {
      "value": [
        "18881 TELEFÔNICA BRASIL S A",
        "28573 CLARO S A ",
        "26599 TELEFÔNICA BRASIL S A",
        "26615 Tim Celular S A ",
        "14593 Space Exploration Technologies Corporation",
        "27699 TELEFÔNICA BRASIL S A"
      ]
    },
    "query_sql": "\nSELECT\n  ISPname AS text,\nFROM `${dataset}.access_ndt7_cached_histograms`(\"MinRTT\", \"${anchor:regex}\", ${ISPcount})\nGROUP BY ISPrank, ISPname\nORDER BY ISPrank\n",
    "default_select": "all"
  },
  {
    "name": "metrics",
    "type": "custom",
    "label": "Metrics",
    "description": "Select which metrics to plot.",
    "hide": 0,
    "multi": true,
    "options": [
      { "text": "MeanThroughputMbps", "value": "MeanThroughputMbps" },
      { "text": "MinRTT", "value": "MinRTT" },
      { "text": "linearMinRTT", "value": "linearMinRTT" },
      { "text": "LossRate", "value": "LossRate" }
    ],
    "current": {
      "value": [
        "MeanThroughputMbps"
      ]
    },
    "query_sql": null
  },
  {
    "name": "binSize",
    "type": "custom",
    "label": "",
    "description": "Re-scale the pdf X axes to the specified number of bins per decade, to smooth Y axis noise.   The raw data is 50 bins per decade.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "50", "value": "50" },
      { "text": "25", "value": "25" },
      { "text": "10", "value": "10" },
      { "text": "5", "value": "5" },
      { "text": "2", "value": "2" },
      { "text": "1", "value": "1" }
    ],
    "current": { "value": "50" },
    "query_sql": "50, 25, 10, 5, 2, 1"
  },
  {
    "name": "xAxis",
    "type": "custom",
    "label": "",
    "description": "Select time granularity for some experimental queries. ",
    "hide": 2,
    "multi": false,
    "options": [
      { "text": "none", "value": "none" },
      { "text": "day", "value": "day" },
      { "text": "hour", "value": "hour" }
    ],
    "current": { "value": "none" },
    "query_sql": "none, day, hour"
  },
  {
    "name": "table_style",
    "type": "custom",
    "label": "Table",
    "description": "Summary table display style.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "none", "value": "none" },
      { "text": "Summary", "value": "Summary" },
      { "text": "Verbose", "value": "Verbose" }
    ],
    "current": { "value": "none" },
    "query_sql": "false, true"
  },
  {
    "name": "table_field",
    "type": "custom",
    "label": "Table Field",
    "description": "Select metric displayed in the table.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "MinRTT", "value": "MinRTT" },
      { "text": "MeanThroughputMbps", "value": "MeanThroughputMbps" },
      { "text": "LossRate", "value": "LossRate" },
      { "text": "linearMinRTT", "value": "linearMinRTT" }
    ],
    "current": { "value": "MeanThroughputMbps" },
    "query_sql": "MinRTT, MeanThroughputMbps, uploadMeanThroughputMbps, LossRate, linearMinRTT, linearMSS, RTO, fineMeanThroughputMbps"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))

# sites= and ISPs= param handling.
# sites= pre-selects servers by site code (e.g. sites=lga04,lga05).
# ISPs= pre-selects ISPs by AS number (e.g. ISPs=7922,8030).
# If sites= is present but anchor= is not, derive anchor from first site code.
_sites_param = [s.strip() for s in url_params.get('sites', '').split(',') if s.strip()]
_isp_asns    = [s.strip() for s in url_params.get('ISPs',  '').split(',') if s.strip()]
if _sites_param and 'anchor' not in url_params:
    url_params['anchor'] = _sites_param[0][:3]
if _sites_param:
    url_params['region'] = _sites_param


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---

# End date + duration — ignored when method=cached.
# End defaults to the most recent Sunday (UTC); duration defaults to 7 days.
_today_utc  = datetime.now(timezone.utc).date()
_days_back  = (_today_utc.weekday() + 1) % 7
_end_date   = _today_utc - timedelta(days=_days_back)
w_to       = widgets.DatePicker(value=_end_date, description='End (UTC)',
                                style={"description_width": "90px"})
w_duration = widgets.Dropdown(
    options=[("1 day", 1), ("7 days", 7), ("28 days", 28), ("30 days", 30)],
    value=7, description='Duration',
    style={"description_width": "90px"},
)
w_from = None

# Date row: start/end range (exp) or end + duration (otherwise). Empty when the
# flavor has no date pickers.
if w_from is not None:
    _date_row = widgets.HBox([w_from, w_to], layout=widgets.Layout(margin='2px 0'))
elif w_to is not None and w_duration is not None:
    _date_row = widgets.HBox([w_to, w_duration], layout=widgets.Layout(margin='2px 0'))
else:
    _date_row = widgets.HTML('')

# The method (or methodsrc, in exp) selector drives date-picker visibility; the
# date row is spliced into the controls column right after it.
_method_var = 'methodsrc' if 'methodsrc' in [v['name'] for v in VARIABLES] else 'method'
# Put endDate + duration on one row (duration second) where both exist (fleet).
_var_names = [v['name'] for v in VARIABLES]
_hgroups = [['endDate', 'duration']] if 'endDate' in _var_names and 'duration' in _var_names else []
ctrl = Controls(VARIABLES, client, presets=url_params,
                asn_presets={'ClientISP': _isp_asns} if _isp_asns else None,
                after={_method_var: _date_row}, hgroups=_hgroups)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")
_date_label = widgets.HTML('')   # filled from query results after Run

# Hide the date row when the backend token is 'cached'; show it otherwise.
_method_w = ctrl.widgets.get(_method_var)
def _toggle_date_row(*_):
    _is_cached = str(getattr(_method_w, 'value', '')).split('-')[0] == 'cached'
    _date_row.layout.display = 'none' if _is_cached else ''
if _method_w is not None:
    _method_w.observe(_toggle_date_row, names='value')
_toggle_date_row()

# "Extra rows" (if present) is shown only when the selected servers span more
# than one metro (distinct 3-letter IATA prefixes of the site codes).
_extra_w   = ctrl.widgets.get('extra_rows')
_servers_w = ctrl.widgets.get('region')
def _toggle_extra_rows(*_):
    _sel = _servers_w.value if _servers_w is not None else ()
    _metros = {str(s)[:3] for s in _sel}
    _row = getattr(_extra_w, 'widget', _extra_w)
    _row.layout.display = '' if len(_metros) > 1 else 'none'
if _extra_w is not None and _servers_w is not None:
    _servers_w.observe(_toggle_extra_rows, names='value')
    _toggle_extra_rows()


In [ ]:
# --- Panels (converted from the dashboard) ---
SUMMARY_PANELS  = [
  {
    'id': 20,
    'title': 'Summary Statistics for the top ISPs in $anchor',
    'type': 'table',
    'sql': r"""
SELECT level, Metric, Sites, tests, pct, KSdistance,
  KSoutlier AS KS_breadcrumb, Spread, SPoutlier AS SP_Breadcrumb, ISPname
FROM (
  SELECT * EXCEPT (ISPname),
  data AS metric,
  percent AS pct,
  IF (level like "Regional Summary",
    FORMAT ("%t summary", REGEXP_REPLACE(ISPname, '^0 ', "")),
    ISPname
    ) AS ISPname,
  -- FROM `${dataset}.cached_metro_report`("MinRTT", "${site:regex}", ${ISPcount})
  FROM `${dataset}.regional_report` ("${method}","${xAxis}", ${binSize}, "${table_field}",
    DATE(REGEXP_EXTRACT("${__from:date:iso}", '[0-9]{4}-[0-9]{2}-[0-9]{2}')),
    DATE(REGEXP_EXTRACT("${__to:date:iso}", '[0-9]{4}-[0-9]{2}-[0-9]{2}')),
    "^(${region:regex})", "^(${ClientISP:regex})", ${ISPcount})
  WHERE ${verbose} OR level like "Regional Summary" OR level LIKE "ISP%%"  
)
""".strip(),
    'layout': {},
    'skip_if_field_none': False,
  },
]
METRIC_LAYOUTS  = json.loads(r"""{
  "MeanThroughputMbps": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.3
    ],
    "gridcolor": "#333"
  },
  "MinRTT": {
    "type": "log",
    "autorange": false,
    "range": [
      -0.3,
      3.0
    ],
    "gridcolor": "#333"
  },
  "linearMinRTT": {
    "type": "linear",
    "autorange": false,
    "range": [
      0,
      300
    ],
    "gridcolor": "#333"
  },
  "LossRate": {
    "type": "log",
    "autorange": true,
    "gridcolor": "#333"
  }
}""")
REPEAT_VAR      = 'ClientISP'

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    to_dt   = (datetime.combine(w_to.value, time(), tzinfo=timezone.utc)
               if w_to and w_to.value else datetime.now(timezone.utc))
    if w_from is not None:
        from_dt = (datetime.combine(w_from.value, time(), tzinfo=timezone.utc)
                   if w_from.value else to_dt - timedelta(days=7))
    else:
        from_dt = to_dt - timedelta(days=(w_duration.value if w_duration else 7))
    cache = {}

    def query(sql):
        if sql not in cache:
            cache[sql] = rt.run_query(client, sql)
        return cache[sql]

    out.clear_output(wait=True)
    with out:
        # Default to "Summary" when table_style is absent (e.g. fleet dashboard).
        table_style = ctx.get("table_style",
                               "Summary" if SUMMARY_PANELS else "none")
        if table_style != "none":
            _sctx = dict(ctx)
            if "table_style" in ctx:
                # Prod: map table_style → verbose flag expected by regional_report SQL.
                _sctx["verbose"] = "true" if table_style == "Verbose" else "false"
            # Other flavors (barchart, fleet) pass verbose directly from ctx.
            for p in SUMMARY_PANELS:
                display(Markdown("### " + qb.interpolate(p["title"], ctx)))
                sql = qb.interpolate(p["sql"], _sctx, from_dt=from_dt, to_dt=to_dt)
                try:
                    _df = query(sql)
                except Exception as exc:
                    display(HTML(f"<pre>query failed: {exc}</pre>"))
                    _df = None
                if _df is not None:
                    if p.get("type") == "barchart":
                        import ipywidgets as _ipyw
                        _link = _ipyw.HTML(
                            value='<p style="color:var(--jp-content-font-color1,#212121);font-size:12px">'
                                  '&#8592; click a bar to open Regional Details</p>'
                        )
                        _fw = rt.metro_barchart_clickable(
                            _df, isp_count=ctx.get("ISPcount", "5"),
                            link_widget=_link)
                        _fw._config = {"responsive": False}
                        display(_fw)
                        display(_link)
                    else:
                        display(HTML(
                            '<div style="height:500px;overflow:auto">'
                            + rt.to_html_sticky(_df, index=False, na_rep="")
                            + '</div>'
                        ))

        repeats = ctx.get(REPEAT_VAR) or []
        if isinstance(repeats, str):
            repeats = [repeats]

        selected_metrics = ctx.get("metrics") or []
        if isinstance(selected_metrics, str):
            selected_metrics = [selected_metrics]

        # One Python call per selected metric fetches data for all client ISPs.
        bulk_by_metric = {}
        _site_regex = qb.format_regex(ctx.get("region") or [])
        _isp_regex  = rt.asn_regex(repeats)
        # "Extra rows" pads the BQ row count only when the selected servers span
        # more than one metro; it is not used for the ISP selection or display.
        _servers = ctx.get("region") or []
        if isinstance(_servers, str):
            _servers = [_servers]
        _metros = {s[:3] for s in _servers}
        _extra_rows = int(ctx.get("extra_rows") or 0) if len(_metros) > 1 else 0
        _isp_count = int(ctx.get("ISPcount", 10)) + _extra_rows
        for metric in selected_metrics:
            try:
                bulk_by_metric[metric] = rt.fetch_histograms(
                    client,
                    method=ctx.get("method", "cached"),
                    field=metric,
                    site_regex=_site_regex,
                    isp_count=_isp_count,
                    bin_size=int(ctx.get("binSize", 50)),
                    x_axis=ctx.get("xAxis", "none"),
                    from_dt=from_dt,
                    to_dt=to_dt,
                    isp_regex=_isp_regex,
                    dataset=ctx.get("dataset",
                                    "mlab-collaboration.mm_preproduction"),
                )
            except Exception as exc:
                bulk_by_metric[metric] = exc

        # Update cached date range label from metroStart/metroEnd in query results.
        for _mdf in bulk_by_metric.values():
            if isinstance(_mdf, pd.DataFrame) and 'metroStart' in _mdf.columns:
                _s = pd.to_datetime(_mdf['metroStart'].dropna().min()).date()
                _e = pd.to_datetime(_mdf['metroEnd'].dropna().max()).date()
                _date_label.value = (
                    '<div style="font-size:12px;color:grey;margin:2px 0">'
                    '<b>Cached data:</b> ' + str(_s) + ' – ' + str(_e) + '</div>')
                break

        for value in repeats:
            asn = str(value).split()[0]
            display(HTML(f"<h3>{REPEAT_VAR}: {value}</h3>"))
            figs = []
            for metric in selected_metrics:
                df_all = bulk_by_metric.get(metric)
                if isinstance(df_all, Exception):
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{df_all}</pre>"))
                    continue
                df = df_all[df_all["ISPname"].str.startswith(asn + " ")]
                try:
                    fig = rt.plotly_combined_figure(
                        df, {"xaxis": METRIC_LAYOUTS.get(metric, {})}, title=metric,
                        sites=_servers)
                    figs.append(go.FigureWidget(fig))
                except Exception as exc:
                    figs.append(widgets.HTML(f"<b>{metric}</b><pre>{exc}</pre>"))
            if figs:
                display(widgets.HBox(figs, layout=widgets.Layout(flex_flow="row wrap")))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, 'Selector Diagnostics')
        _diag_acc.selected_index = None   # collapsed by default
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
# _date_row is inserted inside ctrl.box (right after the method selector) by
# Controls(after=...); only the status label, Run button, and output remain here.
display(widgets.VBox([ctrl.box, w_run, out, _date_label]))
